# 7.2: Decoder block and the complete model


In [ ]:
import math
import torch
import torch.nn as nn

class EncoderBlock(nn.Module):                       # post-norm, as in Section 7.1
    def __init__(self, d, h, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, h, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d))
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, src_pad):
        x = self.norm1(x + self.drop(self.attn(x, x, x, key_padding_mask=src_pad, need_weights=False)[0]))
        return self.norm2(x + self.drop(self.ffn(x)))

class DecoderBlock(nn.Module):
    def __init__(self, d, h, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d, h, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d, h, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d))
        self.norm1, self.norm2, self.norm3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)
    def forward(self, y, memory, causal, tgt_pad, src_pad):
        sa = self.self_attn(y, y, y, attn_mask=causal, key_padding_mask=tgt_pad, need_weights=False)[0]
        y = self.norm1(y + self.drop(sa))
        ca = self.cross_attn(y, memory, memory, key_padding_mask=src_pad, need_weights=False)[0]
        y = self.norm2(y + self.drop(ca))
        return self.norm3(y + self.drop(self.ffn(y)))

def sinusoidal_pe(n, d):
    pos = torch.arange(n)[:, None]
    omega = 10000 ** (-torch.arange(0, d, 2) / d)
    pe = torch.zeros(n, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * omega), torch.cos(pos * omega)
    return pe

class Transformer(nn.Module):
    def __init__(self, V, d=512, h=8, d_ff=2048, N=6, dropout=0.1, max_len=512, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(V, d)                               # shared by source, target, output
        self.register_buffer("pe", sinusoidal_pe(max_len, d))
        self.enc = nn.ModuleList(EncoderBlock(d, h, d_ff, dropout) for _ in range(N))
        self.dec = nn.ModuleList(DecoderBlock(d, h, d_ff, dropout) for _ in range(N))
        self.drop, self.d, self.pad_id = nn.Dropout(dropout), d, pad_id

    def embed(self, tokens):
        return self.drop(self.emb(tokens) * math.sqrt(self.d) + self.pe[: tokens.size(1)])

    def encode(self, src):
        src_pad = src == self.pad_id
        h = self.embed(src)
        for block in self.enc:
            h = block(h, src_pad)
        return h, src_pad

    def decode(self, tgt_in, memory, src_pad):
        n = tgt_in.size(1)
        causal = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)   # True = masked
        g = self.embed(tgt_in)
        for block in self.dec:
            g = block(g, memory, causal, tgt_in == self.pad_id, src_pad)
        return g @ self.emb.weight.T                                # tied output projection

    def forward(self, src, tgt_in):
        memory, src_pad = self.encode(src)
        return self.decode(tgt_in, memory, src_pad)                 # (B, n, V) logits

torch.manual_seed(0)
d, h, d_ff = 512, 8, 2048
blk = DecoderBlock(d, h, d_ff).eval()
ref = nn.TransformerDecoderLayer(d, h, d_ff, batch_first=True).eval()
with torch.no_grad():
    ref.self_attn.load_state_dict(blk.self_attn.state_dict())
    ref.multihead_attn.load_state_dict(blk.cross_attn.state_dict())
    ref.linear1.load_state_dict(blk.ffn[0].state_dict())
    ref.linear2.load_state_dict(blk.ffn[3].state_dict())
    for i in (1, 2, 3):
        getattr(ref, f"norm{i}").load_state_dict(getattr(blk, f"norm{i}").state_dict())
y, mem = torch.randn(2, 5, d), torch.randn(2, 7, d)
causal = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
print(torch.allclose(blk(y, mem, causal, None, None), ref(y, mem, tgt_mask=causal), atol=1e-5))  # True

model = Transformer(V=37000).eval()
src = torch.randint(3, 37000, (2, 11))
tgt_in = torch.randint(3, 37000, (2, 9))
print(model(src, tgt_in).shape)                                    # (2, 9, 37000)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
